In [1]:
##Load libraries
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
from dateutil import parser

In [ ]:
##Import data from directory
#Set file location directory path
file_path = "WhiteHouseLogs/"

#Get list of all files from the directory using glob
file_list = glob.glob(os.path.join(file_path, "*.csv"))

#Create a renaming map due to naming differences in the datasets
renaming_map = {
    "Full Name": "full_name",
    "First Name": "first_name",
    "Last Name": "last_name",
    "Middle Initial": "middle_initial",
    "NAMELAST": "last_name",
    "NAMEFIRST": "first_name",
    "NAMEMID": "middle_initial",
    "UIN": "uin",
    "BDGNBR": "badge_number",
    "Access Type": "access_type",
    "ACCESS_TYPE": "access_type",
    "TOA": "time_of_arrival",
    "TOD": "time_of_departure",
    "POA": "poa",
    "POD": "pod",
    "Appointment Made Date": "appt_made_date",
    "Appointment Start Date": "appt_start_date",
    "Appointment End Date": "appt_end_date",
    "Appointment Cancel Date": "appt_cancel_date",
    "APPT_MADE_DATE": "appt_made_date",
    "APPT_START_DATE": "appt_start_date",
    "APPT_END_DATE": "appt_end_date",
    "APPT_CANCEL_DATE": "appt_cancel_date",
    "Total People": "total_people",
    "TOTAL_PEOPLE": "total_people",
    "Last Updated By": "last_updated_by",
    "LAST_UPDATEDBY": "last_updated_by",
    "Last Entry Date": "last_entry_date",
    "LASTENTRYDATE": "last_entry_date",
    "POST": "post",
    "TERMINAL_SUFFIX": "terminal_suffix",
    "RELEASEDATE": "release_date",
    "Visitee First Name": "visitee_first_name",
    "Visitee Last Name": "visitee_last_name",
    "VISITEE_NAMEFIRST": "visitee_first_name",
    "VISITEE_NAMELAST": "visitee_last_name",
    "Meeting Location": "meeting_location",
    "Meeting Room": "meeting_room",
    "MEETING_LOC": "meeting_location",
    "MEETING_ROOM": "meeting_room",
    "CALLER_ROOM": "caller_room",
    "Caller First Name": "caller_first_name",
    "Caller Last Name": "caller_last_name",
    "CALLER_NAME_FIRST": "caller_first_name",
    "CALLER_NAME_LAST": "caller_last_name",
}

#Load data, rename columns, and combine data
#Create empty dataframe to append each renamed csv dataset to
dfs = []

#Load and clean each csv file and store the data in the dataframe of dataframes
for file in file_list:
    df = pd.read_csv(file, dtype=str)
    df = df.rename(columns=renaming_map)
    dfs.append(df)

#Combine the dataframes
combined_dataset = pd.concat(dfs, ignore_index=True)

#Convert date columns from string to datetime objects
#Date column list
date_columns = [
    "appt_start_date", "appt_end_date", "appt_made_date",
    "appt_cancel_date", "last_entry_date", "release_date"
]

#Convert columns to datetimes
for column in date_columns:
    if column in combined_dataset.columns:
        combined_dataset[column] = combined_dataset[column].apply(
            lambda x: parser.parse(x) if pd.notna(x) and str(x).strip() != "" else pd.NaT
        )

#Create a date column (not datetime) to use where needed in my visuals based on the "appt_start_date" column
    if "appt_start_date" in combined_dataset.columns:
        combined_dataset["date"] = combined_dataset["appt_start_date"].dt.date

#Create a keyvalue to use for distinct visitor identification and counts
combined_dataset["full_name"] = (
    combined_dataset["first_name"].fillna("") + " " +
    combined_dataset["middle_initial"].fillna("") + " " +
    combined_dataset["last_name"].fillna("")
)


In [ ]:
#Visuals build out
#Set theme for all visuals using seaborn
sns.set_theme(style="whitegrid")

In [ ]:
#Creating 1st Visual - Line Chart
#Prepare the data
#Group by date and count unique visitors
daily_counts = (
    combined_dataset.groupby("date")["full_name"]
    .nunique()
    .reset_index(name="unique_visitors")
)

#Sort by date
daily_counts = daily_counts.sort_values("date")

#Line Chart creation
plt.figure(figsize=(14, 6))

plt.plot(
    daily_counts["date"],
    daily_counts["unique_visitors"],
    color="#1f77b4",
    linewidth=1.8
)

plt.title(
    "Daily Unique Visitors to the White House\nHigh Variability Creates Operational Risk",
    fontsize=16,
    pad=20
)

plt.xlabel("Date", fontsize=12)
plt.ylabel("Unique Visitors", fontsize=12)

#Visual clean up
plt.xticks(rotation=45)
plt.tight_layout()
sns.despine()

# --- Add subtle emphasis on spikes (Gestalt: emphasis) ---
# Highlight top 5 spike days
top_spikes = daily_counts.nlargest(10, "unique_visitors")

plt.scatter(
    top_spikes["date"],
    top_spikes["unique_visitors"],
    color="red",
    s=5,
    zorder=5,
    label="High-Volume Spike"
)

plt.legend(frameon=False)

plt.savefig("daily_visitors.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Creating 2nd Visual - Bar Chart

#Prepare the monthly data
combined_dataset["month"] = combined_dataset["appt_start_date"].dt.to_period("M")

monthly_counts = (
    combined_dataset.groupby("month")["full_name"]
    .nunique()
    .reset_index(name="unique_visitors")
)

#Convert Period to Timestamp for plotting
monthly_counts["month"] = monthly_counts["month"].dt.to_timestamp()

#Create my plot
plt.figure(figsize=(14, 6))

plt.bar(
    monthly_counts["month"],
    monthly_counts["unique_visitors"],
    color="#1f77b4",
    width=20 
)

plt.title(
    "Monthly Unique Visitors to the White House\nSeasonal Patterns and Operational Load",
    fontsize=16,
    pad=20
)

plt.xlabel("Month", fontsize=12)
plt.ylabel("Unique Visitors", fontsize=12)

plt.xticks(rotation=45)
sns.despine()
plt.tight_layout()

plt.savefig("monthly_unique_visitors.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Creating 3rd Visual - Stacked Bar Chart

#Create visitee full name from first and last name columns for use in graph
combined_dataset["visitee_name"] = (
    combined_dataset["visitee_first_name"].fillna("") + " " +
    combined_dataset["visitee_last_name"].fillna("")
).str.strip()

#Remove empty strings
combined_dataset["visitee_name"] = combined_dataset["visitee_name"].replace("", pd.NA)

#Set the category to stack by
category = "visitee_name"

#Limit to top 10
top_categories = (
    combined_dataset[category]
    .value_counts()
    .nlargest(10)
    .index
)

combined_dataset["category_group"] = combined_dataset[category].where(
    combined_dataset[category].isin(top_categories),
    "Other"
)

monthly_stacked = (
    combined_dataset.groupby(["month", "category_group"])["full_name"]
    .nunique()
    .reset_index(name="unique_visitors")
)

#Pivoting my data for stacked bar format
pivot_data = monthly_stacked.pivot(
    index="month",
    columns="category_group",
    values="unique_visitors"
).fillna(0)

#Creating my plot
plt.figure(figsize=(16, 7))

pivot_data.plot(
    kind="bar",
    stacked=True,
    figsize=(16, 7),
    colormap="tab20"
)

plt.title(
    "Monthly Unique Visitors by Visitee Category\nUnderstanding Composition of Visitor Volume",
    fontsize=16,
    pad=20
)

plt.xlabel("Month", fontsize=12)
plt.ylabel("Unique Visitors", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
sns.despine()

plt.savefig("monthly_unique_visitors_by_visitee.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Creating 4th Visual - Scatterplot

#Prepare daily data
combined_dataset["date"] = combined_dataset["appt_start_date"].dt.date

daily_data = (
    combined_dataset.groupby("date")
    .agg(
        unique_visitors=("full_name", "nunique"),
        total_appointments=("appt_start_date", "count")
    )
    .reset_index()
)

#Creating a month/year dimension with color
daily_data["month"] = pd.to_datetime(daily_data["date"]).dt.to_period("M").astype(str)

#Creating the plot
plt.figure(figsize=(12, 7))

scatter = sns.scatterplot(
    data=daily_data,
    x="total_appointments",
    y="unique_visitors",
    hue="month",
    palette="viridis",
    alpha=0.8,
    s=60
)

plt.title(
    "Relationship Between Daily Appointments and Unique Visitors\nIdentifying Drivers of High-Volume Days",
    fontsize=16,
    pad=20
)

plt.xlabel("Total Appointments", fontsize=12)
plt.ylabel("Unique Visitors", fontsize=12)
plt.legend(title="Month", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
sns.despine()

plt.savefig("appointment_vs_visitors.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Creating 5th Visual - 7 Day Rolling Average

#Creating datetime for rolling data window
daily_counts["date"] = pd.to_datetime(daily_counts["date"])

#Computing rolling average
daily_counts["rolling_7day"] = (
    daily_counts["unique_visitors"]
    .rolling(window=7, center=False)
    .mean()
)

#Creating chart
plt.figure(figsize=(14, 6))

plt.plot(
    daily_counts["date"],
    daily_counts["rolling_7day"],
    color="#1f77b4",
    linewidth=2.5,
    label="7-Day Rolling Average"
)

plt.title(
    "7-Day Rolling Average of Daily Unique Visitors\nRevealing Underlying Trends Beyond Daily Volatility",
    fontsize=16,
    pad=20
)

plt.xlabel("Date", fontsize=12)
plt.ylabel("Unique Visitors (7-Day Avg)", fontsize=12)
plt.legend()
plt.tight_layout()
sns.despine()

plt.savefig("7day_rolling.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Creating 6th Visual - Step Chart
#Preparing data
daily_counts["date"] = pd.to_datetime(daily_counts["date"])

#Creating threshold for step plot
threshold = daily_counts["unique_visitors"].quantile(0.90)

#Creating column for threshold distinction
daily_counts["above_threshold"] = (daily_counts["unique_visitors"] > threshold).astype(int)

#Create chart
plt.figure(figsize=(14, 5))

plt.step(
    daily_counts["date"],
    daily_counts["above_threshold"],
    where="post",
    linewidth=2,
    color="#d62728"
)

plt.title(
    "High-Volume Threshold Crossings (90th Percentile)\nIdentifying Operationally Critical Days",
    fontsize=16,
    pad=20
)

plt.xlabel("Date", fontsize=12)
plt.ylabel("Above Threshold (1 = Yes)", fontsize=12)
plt.yticks([0, 1], ["No", "Yes"])
plt.tight_layout()
sns.despine()

plt.savefig("threshold_crossings.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import sys
sys.executable